# Snowdon Structural 2x3 - Initial Setup and IFC Type Summary

This notebook follows the same setup style as the Ifc4 SampleHouse transform notebook.

It does three things first:
- Reads both COBie Excel files from the COBie folder.
- Links to Snowdon+Towers+Sample+Structural2x3.json from JSON Whole Model.
- Prints a summary table of IFC Type counts and lists the IFC Type names found in the JSON.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

# ============================================================
# SETUP & PATHS
# ============================================================

workspace_root = Path.cwd().parent
source_json_path = workspace_root / "JSON Whole Model" / "Snowdon+Towers+Sample+Structural2x3.json"
excel_ef_path = workspace_root / "COBie" / "Uniclass2015_EF_v1_16.xlsx"
excel_pr_path = workspace_root / "COBie" / "Uniclass2015_Pr_v1_41.xlsx"

assert source_json_path.exists(), f"JSON not found: {source_json_path}"
assert excel_ef_path.exists(), f"Excel not found: {excel_ef_path}"
assert excel_pr_path.exists(), f"Excel not found: {excel_pr_path}"

print(f"Source JSON: {source_json_path}")
print(f"COBie EF Excel: {excel_ef_path}")
print(f"COBie Pr Excel: {excel_pr_path}")

# ============================================================
# READ BOTH COBie EXCEL FILES
# ============================================================

ef_df = pd.read_excel(excel_ef_path, sheet_name=0)
pr_df = pd.read_excel(excel_pr_path, sheet_name=0)

print("\nExcel files loaded:")
print(f"- EF rows: {len(ef_df):,}, columns: {len(ef_df.columns)}")
print(f"- Pr rows: {len(pr_df):,}, columns: {len(pr_df.columns)}")

# ============================================================
# LOAD JSON AND BUILD IFC TYPE SUMMARY
# ============================================================

with source_json_path.open("r", encoding="utf-8") as f:
    source_data = json.load(f)


def normalize_text(value):
    if value is None:
        return ""
    return str(value).strip()


def get_properties(item):
    properties = item.get("Properties", []) if isinstance(item, dict) else []
    return properties if isinstance(properties, list) else []


def get_ifc_type(item):
    properties = get_properties(item)
    for prop in properties:
        if not isinstance(prop, dict):
            continue

        category = normalize_text(prop.get("category")).lower()
        display_name = normalize_text(prop.get("displayName")).lower()
        value = normalize_text(prop.get("value"))

        if category == "item" and display_name == "type":
            return value.upper()

    return ""


summary_rows = []
for item in source_data:
    ifc_type = get_ifc_type(item)
    if not ifc_type:
        continue
    summary_rows.append({"IFC Type": ifc_type})

summary_df = pd.DataFrame(summary_rows)

if summary_df.empty:
    print("\nNo IFC Type values found in the JSON.")
else:
    ifc_type_count_df = (
        summary_df.groupby("IFC Type", dropna=False)
        .size()
        .rename("Count")
        .reset_index()
        .sort_values(by=["Count", "IFC Type"], ascending=[False, True], kind="stable")
        .reset_index(drop=True)
    )

    print("\nIFC Type summary (count by IFC Type):")
    display(ifc_type_count_df)

    unique_types = ifc_type_count_df["IFC Type"].tolist()
    print(f"\nTotal unique IFC Types: {len(unique_types)}")
    print("IFC Type names:")
    for idx, type_name in enumerate(unique_types, start=1):
        print(f"{idx:>2}. {type_name}")

Source JSON: c:\Git\APS-IFC\JSON Whole Model\Snowdon+Towers+Sample+Structural2x3.json
COBie EF Excel: c:\Git\APS-IFC\COBie\Uniclass2015_EF_v1_16.xlsx
COBie Pr Excel: c:\Git\APS-IFC\COBie\Uniclass2015_Pr_v1_41.xlsx

Excel files loaded:
- EF rows: 233, columns: 14
- Pr rows: 8,453, columns: 14

IFC Type summary (count by IFC Type):


,IFC Type,Count
0,IFCSHAPEREPRESENTATION,4113
1,LCIFCREPRESENTATIONHOLDER,1232
2,IFCMAPPEDITEM,1040
3,IFCBEAM,942
4,IFCPOLYLINE,512
5,IFCEXTRUDEDAREASOLID,486
6,IFCSLAB,108
7,IFCELEMENTASSEMBLY,76
8,IFCCOLUMN,54
9,IFCBUILDINGELEMENTPROXY,50



Total unique IFC Types: 23
IFC Type names:
 1. IFCSHAPEREPRESENTATION
 2. LCIFCREPRESENTATIONHOLDER
 3. IFCMAPPEDITEM
 4. IFCBEAM
 5. IFCPOLYLINE
 6. IFCEXTRUDEDAREASOLID
 7. IFCSLAB
 8. IFCELEMENTASSEMBLY
 9. IFCCOLUMN
10. IFCBUILDINGELEMENTPROXY
11. IFCWALLSTANDARDCASE
12. IFCBOOLEANCLIPPINGRESULT
13. LCOAEXGEOMETRY
14. IFCWALL
15. IFCBUILDINGSTOREY
16. IFCTRIMMEDCURVE
17. COMPOSITE PART
18. IFCGRID
19. IFCFOOTING
20. FILE
21. IFCBUILDING
22. IFCPROJECT
23. IFCSITE
